# 09 RL Environment Making

## 목표
Snake 게임 환경으로 넘어가기 전에, 강화학습의 핵심 구성 요소인 State, Action, Reward, Done을 직접 설계해 본다.
이번 실습에서는 6x6 FoodGrid를 만들고, 학습 알고리즘을 붙이기 전에 환경 자체가 올바르게 동작하는지 로그로 검증하는 흐름을 연습한다.


## 1. 6x6 FoodGrid 규칙
- Grid 크기: `6 x 6`
- Agent: `1개`
- Food: `1개`
- 행동(action): `0=상`, `1=하`, `2=좌`, `3=우`
- 상태(state): 절대 좌표 `(ax, ay, fx, fy)`
- 보상(reward) 초기안: Food 획득 `+10`, 벽 충돌 `-10`, 그 외 매 step `-0.1`
- 종료(done): Agent가 벽에 충돌하면 episode 종료
- 학습 전 점검: Random policy로 여러 episode를 실행하면서 `Step`, `Action`, `Next State`, `Reward`, `Done`을 직접 확인한다.

이 작은 환경에서 설계 감각을 잡아 두면, 나중에 Snake처럼 상태와 보상이 더 복잡한 환경으로 확장할 때 훨씬 수월하다.


In [ ]:
import random

import pandas as pd
from IPython.display import display

SEED = 7
GRID_SIZE = 6

ACTION_NAMES = {0: "UP", 1: "DOWN", 2: "LEFT", 3: "RIGHT"}
ACTION_TO_DELTA = {
    0: (-1, 0),
    1: (1, 0),
    2: (0, -1),
    3: (0, 1),
}

SHAPED_REWARD = {"food": 10.0, "wall": -10.0, "step": -0.1}
SPARSE_REWARD = {"food": 1.0, "wall": -1.0, "step": 0.0}


def format_state(state):
    return f"(ax={state[0]}, ay={state[1]}, fx={state[2]}, fy={state[3]})"


In [ ]:
class FoodGridEnv:
    def __init__(self, size=6, reward_config=None, seed=None):
        self.size = size
        self.reward_config = dict(reward_config or SHAPED_REWARD)
        self.rng = random.Random(seed)
        self.agent_pos = None
        self.food_pos = None
        self.steps = 0

    def reset(self, seed=None, agent_pos=None, food_pos=None):
        if seed is not None:
            self.rng.seed(seed)

        ### TODO: episode가 시작될 때 step 카운터를 0으로 초기화하고, agent_pos가 주어지지 않았다면 랜덤 위치에 에이전트를 놓으세요. ###
        ### TODO: food_pos가 주어지지 않았다면 에이전트와 겹치지 않는 위치에 음식을 생성하고, 직접 주어졌다면 에이전트와 겹치지 않는지 검사하세요. ###
        ### TODO: 마지막에 (ax, ay, fx, fy) 형태의 상태를 반환하세요. ###
        self.steps = 0
        self.agent_pos = None
        self.food_pos = None
        return None

    def step(self, action):
        if action not in ACTION_TO_DELTA:
            raise ValueError(f"invalid action: {action}")

        ### TODO: action에 대응하는 (dx, dy)를 꺼내 현재 agent_pos에 더해 다음 좌표(next_x, next_y)를 계산하세요. ###
        ### TODO: 다음 좌표가 Grid 밖이면 벽 충돌로 처리하고, wall 보상과 done=True를 반환하세요. ###
        ### TODO: 충돌이 아니라면 agent_pos를 갱신하고, food를 먹었는지 확인해 food 보상 또는 step 보상을 계산하세요. ###
        ### TODO: food를 먹었다면 새로운 food를 다시 생성하고, info 딕셔너리에 event / food_eaten 정보를 담으세요. ###
        next_state = None
        reward = None
        done = None
        info = None
        return next_state, reward, done, info

    def _get_state(self):
        ax, ay = self.agent_pos
        fx, fy = self.food_pos
        return (ax, ay, fx, fy)

    def _spawn_food(self):
        candidates = [
            (x, y)
            for x in range(self.size)
            for y in range(self.size)
            if (x, y) != self.agent_pos
        ]
        return self.rng.choice(candidates)

    def render(self):
        grid = [[" . " for _ in range(self.size)] for _ in range(self.size)]
        ax, ay = self.agent_pos
        fx, fy = self.food_pos
        grid[ax][ay] = " A "
        grid[fx][fy] = " F "
        print(f"step={self.steps}, state={format_state(self._get_state())}")
        for row in grid:
            print("".join(row))


In [ ]:
# TODO를 채운 뒤 실행하세요.
env = FoodGridEnv(size=GRID_SIZE, reward_config=SHAPED_REWARD, seed=SEED)
initial_state = env.reset(seed=SEED)
print("Initial state:", initial_state)
# env.render()


## 2. 환경 검증: Random Policy 실행
학습 알고리즘을 연결하기 전에, 먼저 환경이 원하는 규칙대로 움직이는지 확인해야 한다.
가장 쉬운 방법은 random action으로 몇 episode를 돌려 보면서 `Step`, `Action`, `Next State`, `Reward`, `Done` 로그를 직접 읽는 것이다.

아래 루프에서는 3개 episode를 실행하고, 각 step의 전이를 표와 콘솔 로그로 함께 확인한다.


In [ ]:
def run_random_policy(env, episodes=3, max_steps=15, seed=SEED, verbose=True):
    rng = random.Random(seed)
    step_logs = []
    episode_rows = []

    for episode in range(1, episodes + 1):
        state = env.reset(seed=seed + episode)
        episode_reward = 0.0
        foods_eaten = 0
        wall_collisions = 0

        if verbose:
            print(f"\n[random] episode={episode}, start_state={state}")

        for step in range(1, max_steps + 1):
            ### TODO: 0, 1, 2, 3 중 하나를 랜덤하게 뽑아 action 변수에 저장하세요. ###
            action = None

            ### TODO: 선택한 action을 env.step(action)에 넣어 next_state, reward, done, info를 받아오세요. ###
            next_state, reward, done, info = None, None, None, {"event": "TODO", "food_eaten": False}

            if reward is not None:
                episode_reward += reward
                foods_eaten += int(info["food_eaten"])
                wall_collisions += int(info["event"] == "wall_collision")

            row = {
                "episode": episode,
                "step": step,
                "state": state,
                "action": ACTION_NAMES.get(action, "TODO"),
                "next_state": next_state,
                "reward": reward,
                "done": done,
                "event": info["event"],
            }
            step_logs.append(row)

            if verbose:
                print(
                    f"  step={step:02d}, state={state}, action={ACTION_NAMES.get(action, 'TODO'):>5s}, "
                    f"next_state={next_state}, reward={reward}, done={done}, event={info['event']}"
                )
                print("  TODO를 채운 뒤 다시 실행해 보세요.")

            state = next_state
            break

        episode_rows.append(
            {
                "episode": episode,
                "episode_reward": round(episode_reward, 2),
                "foods_eaten": foods_eaten,
                "wall_collisions": wall_collisions,
                "steps": step,
            }
        )

    return pd.DataFrame(step_logs), pd.DataFrame(episode_rows)


In [ ]:
# TODO를 채운 뒤 실행하세요.
random_env = FoodGridEnv(size=GRID_SIZE, reward_config=SHAPED_REWARD, seed=SEED)
random_logs_df, random_summary_df = run_random_policy(random_env, episodes=3, max_steps=15, seed=SEED)

display(random_logs_df)
display(random_summary_df)


## 3. Reward Shaping 실험
같은 Grid와 같은 행동이라도 보상 설계에 따라 에이전트가 받는 학습 신호의 밀도가 달라진다.
여기서는 아래 두 가지 보상을 비교한다.

- Sparse Reward: Food `+1`, 벽 `-1`, 나머지 `0`
- Shaped Reward: Food `+10`, 벽 `-10`, 매 step `-0.1`

학습기를 붙이기 전이라도 random policy 로그를 통해 보상 규모와 분포가 어떻게 달라지는지 감을 잡을 수 있다.


In [ ]:
def evaluate_reward_setting(name, reward_config, episodes=5, max_steps=15, seed=SEED):
    env = FoodGridEnv(size=GRID_SIZE, reward_config=reward_config, seed=seed)
    step_df, summary_df = run_random_policy(
        env,
        episodes=episodes,
        max_steps=max_steps,
        seed=seed,
        verbose=False,
    )
    result = {
        "reward_name": name,
        "episodes": episodes,
        "avg_episode_reward": round(summary_df["episode_reward"].mean(), 2),
        "total_foods": int(summary_df["foods_eaten"].sum()),
        "total_wall_collisions": int(summary_df["wall_collisions"].sum()),
    }
    return result, step_df, summary_df


# TODO를 채운 뒤 실행하세요.
sparse_result, sparse_logs_df, sparse_summary_df = evaluate_reward_setting(
    "Sparse",
    SPARSE_REWARD,
    episodes=5,
    max_steps=15,
    seed=SEED,
)
shaped_result, shaped_logs_df, shaped_summary_df = evaluate_reward_setting(
    "Shaped",
    SHAPED_REWARD,
    episodes=5,
    max_steps=15,
    seed=SEED,
)

comparison_df = pd.DataFrame([sparse_result, shaped_result])
display(comparison_df)
display(sparse_summary_df)
display(shaped_summary_df)


## 4. 상태 표현(State Representation) 설계 토론
같은 환경이라도 상태를 어떻게 정의하느냐에 따라 학습 난이도와 확장성이 크게 달라진다.

- 절대 좌표 `(ax, ay, fx, fy)`
  - 장점: 구현이 가장 단순하고 디버깅이 쉽다.
  - 단점: Grid가 커지거나 Snake 몸통이 생기면 상태가 빠르게 복잡해진다.
- 상대 좌표 `(food_x - agent_x, food_y - agent_y)` 혹은 방향 정보
  - 장점: 목표와의 관계를 직접 담아 일반화에 유리할 수 있다.
  - 단점: 벽, 장애물, 몸통 같은 전역 정보를 충분히 담지 못할 수 있다.
- Grid 전체 One-hot 인코딩
  - 장점: 위치 정보를 풍부하게 담고 Snake처럼 객체가 늘어나는 환경으로 확장하기 좋다.
  - 단점: 차원이 커지고 tabular 방식에는 비효율적이며, 신경망이 사실상 필요해진다.

질문:
Snake 환경으로 확장할 때는 어떤 상태 표현이 더 유리할까?
지금 만든 FoodGrid에서는 무엇이 가장 직관적이고, Snake에서는 왜 다른 선택이 필요해질까?


## 5. 과제 안내
- 필수 과제: 완성된 FoodGrid 환경 코드로 random policy를 `5 episode` 실행하고, `Step / Action / Next State / Reward / Done` 로그 표를 제출하기
- 필수 과제: Reward 설계 `2가지`를 비교하고, 어떤 보상이 더 dense한 학습 신호를 주는지 짧게 분석하기
- 선택 과제: Heuristic agent를 구현해 음식 방향으로 최대한 이동하도록 만들기
- 선택 과제: 상태를 상대 좌표로 바꾼 버전을 추가로 구현하고, 절대 좌표 버전과 차이를 적어 보기
